#### **Multi-Model Spare-Parts Forecasting, Accuracy Benchmarking & Risk Prioritization**

**Objective:**
1. Execute the 6-month holdout backtest (Months 1–30 train, Months 31–36 validation) across four forecasting models:
   - Moving Average (3M)
   - Simple Exponential Smoothing (SES)
   - Holt-Winters Seasonal Exponential Smoothing
   - Croston's Method (SBA Bias-Corrected)
2. Benchmark accuracy using WAPE, MAE, Non-Zero MAPE, and Forecast Bias, selecting the lowest-WAPE champion model per part.
3. Generate 6-month forward demand projections and classify parts into actionable planning tiers (Increase Stock, Maintain Current Stock, Reduce Stock, Review Demand & Master Data).

#### **Imports & Pipeline Execution**

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from src.forecasting import run_forecasting_pipeline
from src.risk_engine import run_risk_pipeline

# Run forecasting and risk scoring pipelines
forecast_outputs = run_forecasting_pipeline(
    processed_dir="../data/processed",
    test_horizon=6,
    forecast_horizon=6,
)
risk_df = run_risk_pipeline(processed_dir="../data/processed")

comp_df = forecast_outputs["model_comparison"]
forecasts_df = forecast_outputs["part_forecasts"]

print(f"Total Model Evaluations: {len(comp_df)} (24 parts x 4 algorithms)")
print(f"Risk Summary Table Shape: {risk_df.shape}")

#### **Aggregate Model Performance Table & Chart**

##### 1. Forecasting Accuracy Benchmarking Across Algorithms
- Compare average and median WAPE, MAE, and Forecast Bias across all four algorithms, and examine which champion models win by demand pattern (`Fast-Moving`, `Slow-Moving`, `Intermittent`, `Lumpy`).

In [ ]:
# Merge demand pattern into model comparison table
comp_enriched = comp_df.merge(
    risk_df[["part_number", "demand_pattern"]], on="part_number", how="left"
)

model_benchmark = (
    comp_enriched.groupby("model_name", as_index=False)
    .agg(
        median_wape_pct=("wape_pct", "median"),
        mean_wape_pct=("wape_pct", "mean"),
        median_mae=("mae", "median"),
        mean_bias_pct=("forecast_bias_pct", "mean"),
        champion_wins=("is_champion", "sum"),
    )
    .round(2)
    .sort_values("median_wape_pct")
)
display(model_benchmark)

# Champion win distribution by demand pattern
champion_by_pattern = (
    comp_enriched[comp_enriched["is_champion"]]
    .groupby(["demand_pattern", "model_name"], as_index=False)
    .size()
    .rename(columns={"size": "part_count"})
)

fig_champ = px.bar(
    champion_by_pattern,
    x="demand_pattern",
    y="part_count",
    color="model_name",
    barmode="group",
    title="Champion Model Selection by Spare-Part Demand Pattern",
    labels={"demand_pattern": "Demand Classification", "part_count": "Number of Parts Won"},
    text_auto=True,
)
fig_champ.update_layout(height=420, template="plotly_white")
fig_champ.show()

#### **Accuracy Target Achievement & Scatter Plot**

##### 2. Fleet-Wide Champion Accuracy & Bias Audit
- Verify how many parts meet the target 75%+ Forecast Accuracy (WAPE < 25%) and inspect where high-error parts are concentrated.

In [ ]:
champions_only = risk_df.copy()
within_target = (champions_only["wape_pct"] <= 25.0).sum()

print(f"Parts meeting WAPE <= 25% target: {within_target} / {len(champions_only)} ({within_target/len(champions_only):.1%})")
print(f"Median Champion WAPE: {champions_only['wape_pct'].median():.2f}%")
print(f"Median Forecast Accuracy (1 - WAPE): {champions_only['forecast_accuracy_pct'].median():.2f}%")
print(f"Median Forecast Bias: {champions_only['forecast_bias_pct'].median():.2f}%")

fig_acc = px.scatter(
    champions_only,
    x="intermittency_rate_pct",
    y="wape_pct",
    color="demand_pattern",
    size="total_spend_aed",
    hover_name="part_name",
    hover_data=["champion_model", "mae", "forecast_bias_pct"],
    title="Champion Forecast Error (WAPE %) vs. Intermittency Rate (%)",
    labels={
        "intermittency_rate_pct": "Intermittency Rate (% Zero-Demand Months)",
        "wape_pct": "Champion WAPE (%)",
    },
)
fig_acc.add_hline(y=25.0, line_dash="dash", line_color="red", annotation_text="25% WAPE Threshold")
fig_acc.update_layout(height=430, template="plotly_white")
fig_acc.show()

#### **Visual Forecast Inspection Function**

##### 3. Part-Level Forecast Visualization
- Inspect historical demand, 6-month holdout backtest predictions, and forward 6-month projections for both a Fast-Moving part (`PRT-005`: Front Disc Brake Pad Set) and an Intermittent part (`PRT-016`: A/C Compressor Assembly).

In [ ]:
def plot_part_forecast(part_num: str):
    sub = forecasts_df[forecasts_df["part_number"] == part_num].sort_values("month")
    part_title = f"{part_num} — {sub['part_name'].iloc[0]} (Model: {sub['selected_model'].iloc[0]} | WAPE: {sub['model_wape_pct'].iloc[0]}%)"

    fig = go.Figure()
    # Historical + Backtest Actuals
    actuals = sub[sub["record_type"].isin(["Historical", "Backtest"])]
    fig.add_trace(
        go.Scatter(
            x=actuals["month"],
            y=actuals["actual_qty"],
            mode="lines+markers",
            name="Actual Demand",
            line=dict(color="#1f77b4", width=2.5),
        )
    )
    # Backtest Validation Predictions
    backtest = sub[sub["record_type"] == "Backtest"]
    fig.add_trace(
        go.Scatter(
            x=backtest["month"],
            y=backtest["forecast_qty"],
            mode="lines+markers",
            name="Backtest Prediction (6M)",
            line=dict(color="#ff7f0e", width=2.5, dash="dash"),
        )
    )
    # Forward 6-Month Forecast
    future = sub[sub["record_type"] == "Forecast"]
    fig.add_trace(
        go.Scatter(
            x=future["month"],
            y=future["forecast_qty"],
            mode="lines+markers",
            name="6-Month Forward Forecast",
            line=dict(color="#2ca02c", width=3, dash="dot"),
        )
    )
    fig.update_layout(
        title=part_title,
        xaxis_title="Month",
        yaxis_title="Units",
        height=400,
        template="plotly_white",
    )
    fig.show()

plot_part_forecast("PRT-005")
plot_part_forecast("PRT-016")

#### **Risk Summary & Action Matrix**

In [ ]:
action_summary = (
    risk_df.groupby("recommended_action", as_index=False)
    .agg(
        part_count=("part_number", "count"),
        next_3m_units=("forecast_next_3m_qty", "sum"),
        next_6m_units=("forecast_next_6m_qty", "sum"),
        next_6m_spend_aed=("forecast_next_6m_spend_aed", "sum"),
    )
    .round(2)
)
display(action_summary)

print("\n--- Top Prioritized Risk & Action Table ---")
display(
    risk_df[
        [
            "part_number",
            "part_name",
            "demand_pattern",
            "risk_tier",
            "risk_score",
            "recommended_action",
            "forecast_next_3m_qty",
            "forecast_next_6m_qty",
            "wape_pct",
            "risk_drivers",
        ]
    ].head(12)
)